Production Ready Pipeline covering file loading, API ingestion, and quality validation



In [ ]:
import pandas as pd
import requests
from pathlib import Path

# Load data with explicit parameters
def load_csv_safe(path, date_cols=None, dtypes=None):
  """Load CSV with all safeguards against silent corruption."""
  return pd.read_csv(
      path,
      encoding='utf-8',
      dtype=dtypes or {},
      parse_dates=date_cols or [],
      na_values=['','NULL','N/A', 'null', 'None', '-', '--', 'n/a'],
      low_memory=False
  )

#Example usage
df = load_csv_safe(
    'messy_ecommerce_customers_10k.csv',
    date_cols=['signup_date','last_login_date','last_purchase_date'],
    dtypes={'customer_id': str, 'first_name': str}
)


# 2. Five Quality Audit Check

def quality_audit(df, name='Dataset'):
  """Run the 5 essential checks on any new dataset."""
  print(f"\n{'='*50}")
  print(f"Quality Audit: {name}")
  print(f"{'='*50}")

  #Check Shape
  print(f"\n1.Shape: {df.shape[0]:,} rows x {df.shape[1]:,} cols")

  #Check Missing Values
  missing = df.isnull().sum()
  missing_pct = (missing / len(df) * 100).round(1)
  has_missing = missing[missing > 0]
  if len(has_missing) > 0:
    print(f"\n2.Missing Values ({len(has_missing)} columns):")
    for col in has_missing.index:
      print(f" {col}: {has_missing[col]:,} ({missing_pct[col]}%)")
  else:
    print(f"\n2. Missing values: None!")

  # 3. Data types
  print(f"\n3.Data types")
  for dtype, count in df.dtypes.value_counts().items():
    print(f" {dtype}: {count} columns")

  # 4. Check Duplicates
  n_dupes = df.duplicated().sum()
  print(f"\n4. Duplicates: {n_dupes:,} ({n_dupes/len(df) * 100}:.1f%)")

  # 5. Value ranges (numerical only)
  print(f"\n5.Value ranges (numeric)")
  for col in df.select_dtypes(include='number').columns[:5]:
    print(f" {col}: [{df[col].min():.2f}, {df[col].max():.2f}]")

quality_audit(df, "Customer Data")



Quality Audit: Customer Data

1.Shape: 10,050 rows x 18 cols

2.Missing Values (4 columns):
 age: 499 (5.0%)
 gender: 303 (3.0%)
 preferred_category: 203 (2.0%)
 customer_satisfaction_score: 805 (8.0%)

3.Data types
 object: 11 columns
 float64: 4 columns
 datetime64[ns]: 2 columns
 bool: 1 columns

4. Duplicates: 50 (0.4975124378109453:.1f%)

5.Value ranges (numeric)
 age: [-48.00, 299.00]
 total_orders: [0.00, 149.00]
 customer_satisfaction_score: [1.00, 14.00]
 cart_abandonment_rate: [0.00, 1.00]


Data Loading: Pandas vs Polars vs DuckDB
Compare modern data tools on the same analytical task

In [ ]:
import pandas as pd
import time

# ============================================
# BENCHMARK: Same operations, different tools
# ============================================

# --- PANDAS (traditional) ---
start = time.time()
df_pd = pd.read_csv('sales.csv',
    dtype={'store_id': str},
    parse_dates=['date'])
# Group by store, compute monthly revenue
result_pd = (df_pd
    .groupby(['store_id', df_pd['date'].dt.month])
    ['revenue']
    .agg(['sum', 'mean', 'count'])
    .reset_index()
)
print(f"Pandas: {time.time()-start:.2f}s, {len(df_pd):,} rows")

# --- POLARS (modern, fast) ---
import polars as pl

start = time.time()
df_pl = pl.read_csv('sales.csv',
    schema_overrides={'store_id': pl.String},
    try_parse_dates=True
)
# Same operation but runs multi-threaded
result_pl = (df_pl
    .group_by([pl.col('store_id'), pl.col('date').dt.month()])
    .agg([
        pl.col('revenue').sum().alias('sum'),
        pl.col('revenue').mean().alias('mean'),
        pl.col('revenue').len().alias('count')
    ])
)
print(f"Polars: {time.time()-start:.2f}s, {len(df_pl):,} rows")

# --- DUCKDB (SQL on files) ---
import duckdb

start = time.time()
result_db = duckdb.sql("""
    SELECT
        store_id,
        MONTH(date) as month,
        SUM(revenue) as total,
        AVG(revenue) as avg_rev,
        COUNT(*) as n_transactions
    FROM read_csv_auto('sales.csv')
    GROUP BY store_id, MONTH(date)
    ORDER BY total DESC
""").df()
print(f"DuckDB: {time.time()-start:.2f}s")

# ============================================
# CONVERT CSV TO PARQUET (do this once!)
# ============================================
df_pd.to_parquet('sales.parquet', index=False)

# Then load Parquet (much faster for selective column reads)
df_fast = pd.read_parquet('sales.parquet',
    columns=['store_id', 'revenue']  # Only needed columns
)
print(f"\nParquet selective load: {len(df_fast):,} rows, 2 columns")

Stratified Sampling and Data Versioning
Build representative samples and track data lineage

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import hashlib
from datetime import datetime

# ============================================
# 1. STRATIFIED SAMPLING FOR IMBALANCED DATA
# ============================================
np.random.seed(42)
n = 10000

# Simulate imbalanced fraud dataset (3% fraud)
df = pd.DataFrame({
    'amount': np.random.exponential(100, n),
    'hour': np.random.randint(0, 24, n),
    'is_fraud': np.random.choice([0, 1], n, p=[0.97, 0.03])
})

print(f"Full dataset: {len(df):,} rows")
print(f"Fraud rate: {df['is_fraud'].mean():.1%}")

# BAD: Random sampling (fraud may be underrepresented)
random_sample = df.sample(1000, random_state=42)
print(f"\nRandom sample fraud rate: {random_sample['is_fraud'].mean():.1%}")

# GOOD: Stratified sampling (preserves fraud proportion)
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        # Preserves class balance
    random_state=42
)
print(f"Stratified train fraud rate: {y_train.mean():.1%}")
print(f"Stratified test fraud rate:  {y_test.mean():.1%}")

# ============================================
# 2. DATA FINGERPRINTING (detect changes)
# ============================================
def data_fingerprint(df):
    """Create a hash fingerprint of a DataFrame."""
    content = pd.util.hash_pandas_object(df).values.tobytes()
    return hashlib.md5(content).hexdigest()[:12]

fingerprint = data_fingerprint(df)
print(f"\nData fingerprint: {fingerprint}")

# ============================================
# 3. DATA COLLECTION METADATA LOG
# ============================================
import json

collection_log = {
    'timestamp': datetime.now().isoformat(),
    'source': 'synthetic_fraud_data',
    'rows': len(df),
    'columns': list(df.columns),
    'fraud_rate': float(df['is_fraud'].mean()),
    'fingerprint': fingerprint,
    'missing_values': int(df.isnull().sum().sum()),
    'duplicates': int(df.duplicated().sum()),
    'splits': {
        'train': len(X_train),
        'test': len(X_test),
        'stratified': True
    }
}

print(f"\nCollection Log:")
print(json.dumps(collection_log, indent=2))